<a href="https://colab.research.google.com/github/Matteo-chf/projet-climat/blob/main/%09notebooks/01_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
# ============================================================
# Projet : Réchauffement climatique dans les villes européennes
# Étape 1 : Extraction (batch layer) - Open-Meteo -> MongoDB Atlas
# À copier cellule par cellule dans Google Colab (# %% = nouvelle cellule)
# Données : Open-Meteo.com (licence CC BY 4.0, usage non commercial)
# ============================================================

# %% Cellule 1 : installation (dans Colab, décommente la ligne)
# !pip install -q "pymongo[srv]" pandas requests

# %% Cellule 2 : configuration
import time
import requests
import pandas as pd
from pymongo import MongoClient
from google.colab import files

In [9]:


URL = "https://archive-api.open-meteo.com/v1/archive"
START, END = "1950-01-01", "2025-12-31"   # 76 années complètes
VARIABLES = ["temperature_2m_max", "temperature_2m_min",
             "temperature_2m_mean", "precipitation_sum"]

# Coordonnées approximatives (lat, lon) : à ajuster/compléter si besoin
CITIES = {
    "Reykjavik": (64.15, -21.94), "Oslo": (59.91, 10.75),
    "Stockholm": (59.33, 18.07), "Helsinki": (60.17, 24.94),
    "Moscou": (55.75, 37.62), "Saint-Petersbourg": (59.93, 30.34),
    "Copenhague": (55.68, 12.57), "Dublin": (53.35, -6.26),
    "Londres": (51.51, -0.13), "Amsterdam": (52.37, 4.90),
    "Berlin": (52.52, 13.41), "Varsovie": (52.23, 21.01),
    "Kiev": (50.45, 30.52), "Bruxelles": (50.85, 4.35),
    "Paris": (48.86, 2.35), "Prague": (50.08, 14.44),
    "Vienne": (48.21, 16.37), "Budapest": (47.50, 19.04),
    "Munich": (48.14, 11.58), "Zurich": (47.37, 8.54),
    "Strasbourg": (48.57, 7.75), "Lyon": (45.76, 4.84),
    "Bordeaux": (44.84, -0.58), "Brest": (48.39, -4.49),
    "Marseille": (43.30, 5.37), "Milan": (45.46, 9.19),
    "Bucarest": (44.43, 26.10), "Belgrade": (44.79, 20.45),
    "Sofia": (42.70, 23.32), "Rome": (41.90, 12.50),
    "Madrid": (40.42, -3.70), "Barcelone": (41.39, 2.17),
    "Lisbonne": (38.72, -9.14), "Seville": (37.39, -5.98),
    "Naples": (40.85, 14.27), "Palerme": (38.12, 13.36),
    "Athenes": (37.98, 23.73), "Istanbul": (41.01, 28.98),
    "Valence": (39.47, -0.38), "Edimbourg": (55.95, -3.19),
}

In [14]:
# %% Cellule 3 : fonction d'extraction (une requête par ville)
def fetch_city(name, lat, lon, max_wait_min=75):
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": START, "end_date": END,
        "daily": ",".join(VARIABLES),
        "timezone": "UTC",
    }
    waited = 0
    while True:
        r = requests.get(URL, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code == 429:
            reason = r.json().get("reason", "")
            if "daily" in reason.lower():
                raise SystemExit("Limite journalière atteinte : reprends demain, "
                                 "les villes déjà insérées sont conservées.")
            if waited >= max_wait_min:
                raise RuntimeError(f"Toujours bloqué pour {name} : {reason}")
            pause = 1 if "minutely" in reason.lower() else 5
            print(f"  429 pour {name} : {reason} -> pause {pause} min (attendu {waited} min)")
            time.sleep(pause * 60)
            waited += pause
            continue
        r.raise_for_status()

In [15]:
# %% Cellule 4 : transformation en documents "ville-année"
def to_year_documents(name, payload):
    daily = pd.DataFrame(payload["daily"])
    daily["time"] = pd.to_datetime(daily["time"])
    daily["year"] = daily["time"].dt.year
    docs = []
    for year, g in daily.groupby("year"):
        docs.append({
            "city": name,
            "year": int(year),
            "latitude": payload["latitude"],
            "longitude": payload["longitude"],
            "elevation": payload.get("elevation"),
            "dates": g["time"].dt.strftime("%Y-%m-%d").tolist(),
            **{v: g[v].tolist() for v in VARIABLES},
            "source": "open-meteo-archive",
        })
    return docs

In [16]:
# %% Cellule 5 : connexion MongoDB Atlas (URI dans les Secrets de Colab)
# Colab > icône clé (Secrets) > ajouter MONGO_URI > activer l'accès au notebook
from google.colab import userdata
client = MongoClient(userdata.get("MONGO_URI"))
client.admin.command("ping")               # lève une erreur si connexion KO
print("Connexion Atlas OK")
col = client["climat_europe"]["raw_climate"]

Connexion Atlas OK


In [24]:
!pip install -q codecarbon

In [25]:
# %% Cellule 6 : boucle batch (≈ 40 requêtes) avec suivi énergétique
from codecarbon import OfflineEmissionsTracker

tracker = OfflineEmissionsTracker(
    country_iso_code="FRA",
    project_name="climat_europe_extraction",
    output_dir=".",
    output_file="emissions.csv",
    measure_power_secs=30,
    log_level="error",
)
tracker.start()

all_docs_count = 0
try:
    for i, (city, (lat, lon)) in enumerate(CITIES.items(), 1):
        if col.count_documents({"city": city}, limit=1):
            print(f"[{i}/{len(CITIES)}] {city} déjà en base, ignorée")
            continue
        payload = fetch_city(city, lat, lon)
        docs = to_year_documents(city, payload)
        col.insert_many(docs)
        all_docs_count += len(docs)
        print(f"[{i}/{len(CITIES)}] {city} : {len(docs)} documents "
              f"(altitude {payload.get('elevation')} m)")
        time.sleep(2)                      # politesse envers l'API gratuite

    col.create_index([("city", 1), ("year", 1)], unique=True)
    print("Terminé :", col.count_documents({}), "documents dans raw_climate")
finally:
    # S'exécute toujours : fin normale, erreur ou interruption manuelle
    emissions_kg = tracker.stop()
    print(f"Émissions estimées : {emissions_kg * 1000:.3f} g CO2e")

[1/40] Reykjavik déjà en base, ignorée
[2/40] Oslo déjà en base, ignorée
[3/40] Stockholm déjà en base, ignorée
[4/40] Helsinki déjà en base, ignorée
[5/40] Moscou déjà en base, ignorée
[6/40] Saint-Petersbourg déjà en base, ignorée
[7/40] Copenhague déjà en base, ignorée
[8/40] Dublin déjà en base, ignorée
[9/40] Londres déjà en base, ignorée
[10/40] Amsterdam déjà en base, ignorée
[11/40] Berlin déjà en base, ignorée
[12/40] Varsovie déjà en base, ignorée
[13/40] Kiev : 76 documents (altitude 169.0 m)
Émissions estimées : 0.004 g CO2e


SystemExit: Limite journalière atteinte : reprends demain, les villes déjà insérées sont conservées.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [21]:
df = pd.read_csv("emissions.csv")
print(df[["timestamp", "duration", "energy_consumed", "emissions"]])
files.download("emissions.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'emissions.csv'

In [26]:
# %% Cellule 7 : contrôle rapide (pour une capture d'écran du README)
doc = col.find_one({"city": "Paris", "year": 2003})
print({k: (v if not isinstance(v, list) else f"liste de {len(v)} valeurs")
       for k, v in doc.items() if k != "_id"})
print("Taille de la collection (Mo) :",
      round(client["climat_europe"].command("collstats", "raw_climate")["size"] / 1e6, 1))

AttributeError: 'NoneType' object has no attribute 'items'

In [27]:
# %% Cellule 8 : récupération du suivi énergétique
df = pd.read_csv("emissions.csv")
print(df[["timestamp", "duration", "energy_consumed", "emissions"]])
files.download("emissions.csv")

             timestamp    duration  energy_consumed  emissions
0  2026-11-13T16:04:35  297.662630         0.003335   0.000187
1  2026-11-13T16:06:23    6.400737         0.000069   0.000004


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>